<a href="https://colab.research.google.com/github/ashvin-a/Learn-RL/blob/main/K1_movesets.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# K1 Walk Policy

Train a robust bipedal walking policy for the **Booster K1** humanoid using:
- **MuJoCo** — physics simulation built from the K1's 22-DOF URDF (12-DOF legs active)
- **PPO** via `stable-baselines3` — on-policy RL with GAE + observation normalisation
- **Weights & Biases** — reward curves, videos, and hyperparameter sweeps

### Joint structure (12 controllable leg DOF)

| Side | Joint | Range (rad) | Max torque (Nm) |
|------|-------|-------------|-----------------|
| L/R  | Hip Pitch  | −3.00 / +2.21 | 30 |
| L/R  | Hip Roll   | ±1.57         | 35 |
| L/R  | Hip Yaw    | ±1.00         | 20 |
| L/R  | Knee Pitch | 0.00 / +2.23  | 40 |
| L/R  | Ankle Pitch| −0.87 / +0.35 | 20 |
| L/R  | Ankle Roll | ±0.345        | 20 |

### Observation space (48 dims)
`base_lin_vel(3) | base_ang_vel(3) | proj_gravity(3) | cmd_vel(3) | q_offset(12) | dq(12) | prev_action(12)`

### Reward components
`lin_vel_tracking + ang_vel_tracking − lin_vel_z − ang_vel_xy − orientation − height + feet_air_time − energy − action_rate`

In [ ]:
# Install dependencies — restart runtime once after this cell if in Colab
%pip install -q mujoco gymnasium "stable-baselines3[extra]" wandb imageio mediapy

In [ ]:
import os, time
import numpy as np
import mujoco
import gymnasium as gym
from gymnasium import spaces
import wandb
from wandb.integration.sb3 import WandbCallback
import imageio

import torch
import torch.nn as nn
import stable_baselines3 as sb3
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import BaseCallback, EvalCallback, CheckpointCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.utils import set_random_seed

print(f"MuJoCo  : {mujoco.__version__}")
print(f"SB3     : {sb3.__version__}")
print(f"PyTorch : {torch.__version__}  |  CUDA: {torch.cuda.is_available()}")

In [ ]:
import os
import wandb

# ── W&B authentication ────────────────────────────────────────────────────────
# Your API key should be set as an environment variable, never hardcoded.
#
#   Option A — env var (recommended for local / Colab terminal):
#       export WANDB_API_KEY=<your-key>
#
#   Option B — Colab Secrets (recommended for notebooks):
#       Runtime → Manage secrets → add WANDB_API_KEY
#
#   Option C — paste key at the wandb.login() prompt below.
#
try:
    from google.colab import userdata as _ud
    _k = _ud.get("WANDB_API_KEY")
    if _k:
        os.environ["WANDB_API_KEY"] = _k
except Exception:
    pass   # not in Colab — use env var or interactive prompt

wandb.login()   # uses WANDB_API_KEY env var; prompts if not set

WANDB_PROJECT   = "k1-walk"
WANDB_ENTITY    = None          # None → auto-detects your logged-in account
EXPERIMENT_NAME = "ppo-k1-12dof"

print(f"W&B project : {WANDB_PROJECT!r}")
print(f"Experiment  : {EXPERIMENT_NAME!r}")

In [ ]:
# ── K1 MuJoCo model (MJCF) ───────────────────────────────────────────────────
# Derived from K1_22dof.urdf — 12-DOF legs only (head & arms removed).
# Inertial data carried over exactly; simple primitive geometries used
# in place of mesh files so no asset directory is required.
# MuJoCo fullinertia order: ixx iyy izz ixy ixz iyz
#   (URDF order:            ixx ixy ixz iyy iyz izz — note the difference)

K1_MJCF = """
<mujoco model="K1">
  <compiler angle="radian" balanceinertia="true" discardvisual="false"/>
  <option timestep="0.005" integrator="RK4" gravity="0 0 -9.81"/>

  <default>
    <joint limited="true" damping="0.5" armature="0.01" stiffness="0"/>
    <geom condim="3" friction="0.8 0.02 0.01" rgba="0.75 0.75 0.75 1"/>
    <motor ctrllimited="true" ctrlrange="-1 1"/>
  </default>

  <asset>
    <texture name="checker" type="2d" builtin="checker" width="512" height="512"
             rgb1="0.85 0.85 0.85" rgb2="0.95 0.95 0.95"/>
    <material name="floor_mat" texture="checker" texrepeat="8 8" reflectance="0.1"/>
    <material name="left_mat"  rgba="0.8 0.35 0.35 1"/>
    <material name="right_mat" rgba="0.35 0.35 0.8 1"/>
    <material name="foot_mat"  rgba="0.2 0.2 0.2 1"/>
  </asset>

  <worldbody>
    <light name="sun" directional="true" diffuse="0.8 0.8 0.8"
           specular="0.2 0.2 0.2" pos="2 1 4" dir="-1 -0.5 -3"/>
    <geom name="floor" type="plane" size="50 50 0.1" material="floor_mat" condim="3"/>

    <body name="Trunk" pos="0 0 0.55">
      <freejoint name="root"/>
      <inertial pos="-0.00434 -0.000655 0.06569" mass="6.50"
                fullinertia="0.096159 0.089512 0.020187 -0.000039 -0.002091 -0.000567"/>
      <site name="imu" pos="0 0 0.065" size="0.01"/>
      <camera name="track" pos="0 -2.5 1.0" xyaxes="1 0 0 0 0.4 1" mode="trackcom"/>
      <geom type="box"      size="0.06 0.09 0.10" pos="0 0 0.10"/>
      <geom type="cylinder" size="0.05 0.05"       pos="0 0 -0.06" euler="1.5708 0 0"/>

      <!-- ════════════ LEFT LEG ════════════ -->
      <body name="left_hip_pitch_link" pos="0 0.096 -0.077">
        <inertial pos="-0.010093 -0.001993 -0.024713" mass="0.69"
                  fullinertia="0.000781 0.000987 0.000604 0.000004 0.000167 0.000008"/>
        <joint name="Left_Hip_Pitch" type="hinge" axis="0 1 0"
               range="-3.0 2.21" damping="0.5" armature="0.01"/>
        <geom type="cylinder" size="0.025 0.02" euler="1.5708 0 0" material="left_mat"/>

        <body name="left_hip_roll_link" pos="0 0 -0.026">
          <inertial pos="0.023585 0.000049 -0.024996" mass="0.13"
                    fullinertia="0.000191 0.000262 0.000150 0.000000 -0.000048 0.000000"/>
          <joint name="Left_Hip_Roll" type="hinge" axis="1 0 0"
                 range="-0.4 1.57" damping="0.5" armature="0.01"/>
          <geom type="cylinder" size="0.036 0.047" euler="0 1.5708 0" material="left_mat"/>

          <body name="left_thigh_link" pos="0.012 0 -0.0485">
            <inertial pos="-0.008474 -0.004049 -0.087906" mass="1.65"
                      fullinertia="0.015786 0.016140 0.001551 0.000096 0.001546 0.000745"/>
            <joint name="Left_Hip_Yaw" type="hinge" axis="0 0 1"
                   range="-1.0 1.0" damping="0.5" armature="0.01"/>
            <geom type="cylinder" size="0.040 0.033" pos="0 0 -0.033" material="left_mat"/>
            <geom type="capsule"  size="0.025" fromto="0 0 -0.066 0 0 -0.117" material="left_mat"/>

            <body name="left_shank_link" pos="-0.014 0 -0.117">
              <inertial pos="-0.00081 0.003146 -0.10922" mass="1.50"
                        fullinertia="0.023681 0.023670 0.001240 0.000035 0.000222 -0.000105"/>
              <joint name="Left_Knee_Pitch" type="hinge" axis="0 1 0"
                     range="0.0 2.23" damping="0.5" armature="0.01"/>
              <geom type="cylinder" size="0.045 0.035" euler="1.5708 0 0" material="left_mat"/>
              <geom type="capsule"  size="0.022" fromto="0 0 -0.02 0 0 -0.245" material="left_mat"/>

              <body name="left_ankle_cross_link" pos="0.0002 0.0002 -0.24519">
                <inertial pos="0 0 0" mass="0.039"
                          fullinertia="0.000002 0.000004 0.000004 0 0 0"/>
                <joint name="Left_Ankle_Pitch" type="hinge" axis="0 1 0"
                       range="-0.87 0.345" damping="0.3" armature="0.005"/>

                <body name="left_foot_link" pos="0 0 0">
                  <inertial pos="0.011662 0.000244 -0.014364" mass="0.494"
                            fullinertia="0.000261 0.001170 0.001213 0.0000003 -0.000115 -0.0000002"/>
                  <joint name="Left_Ankle_Roll" type="hinge" axis="1 0 0"
                         range="-0.345 0.345" damping="0.3" armature="0.005"/>
                  <geom name="left_foot_geom" type="box" size="0.10 0.045 0.020"
                        pos="0.04 0 -0.025" material="foot_mat"
                        condim="3" friction="0.8 0.02 0.01"/>
                  <site name="left_foot_site" pos="0.04 0 -0.025" size="0.005"/>
                </body>
              </body>
            </body>
          </body>
        </body>
      </body>

      <!-- ════════════ RIGHT LEG ════════════ -->
      <body name="right_hip_pitch_link" pos="0 -0.096 -0.077">
        <inertial pos="-0.010065 0.002009 -0.024739" mass="0.69"
                  fullinertia="0.000782 0.000988 0.000604 -0.000004 0.000166 -0.000009"/>
        <joint name="Right_Hip_Pitch" type="hinge" axis="0 1 0"
               range="-3.0 2.21" damping="0.5" armature="0.01"/>
        <geom type="cylinder" size="0.025 0.02" euler="1.5708 0 0" material="right_mat"/>

        <body name="right_hip_roll_link" pos="0 0 -0.026">
          <inertial pos="0.023585 0.000049 -0.024996" mass="0.13"
                    fullinertia="0.000191 0.000262 0.000150 0.000000 -0.000048 0.000000"/>
          <joint name="Right_Hip_Roll" type="hinge" axis="1 0 0"
                 range="-1.57 0.4" damping="0.5" armature="0.01"/>
          <geom type="cylinder" size="0.036 0.047" euler="0 1.5708 0" material="right_mat"/>

          <body name="right_thigh_link" pos="0.012 0 -0.0485">
            <inertial pos="-0.008475 0.004039 -0.087906" mass="1.65"
                      fullinertia="0.015786 0.016140 0.001551 -0.000096 0.001546 -0.000745"/>
            <joint name="Right_Hip_Yaw" type="hinge" axis="0 0 1"
                   range="-1.0 1.0" damping="0.5" armature="0.01"/>
            <geom type="cylinder" size="0.040 0.033" pos="0 0 -0.033" material="right_mat"/>
            <geom type="capsule"  size="0.025" fromto="0 0 -0.066 0 0 -0.117" material="right_mat"/>

            <body name="right_shank_link" pos="-0.014 0 -0.117">
              <inertial pos="-0.000805 -0.003146 -0.109215" mass="1.50"
                        fullinertia="0.023681 0.023670 0.001240 -0.000035 0.000221 0.000105"/>
              <joint name="Right_Knee_Pitch" type="hinge" axis="0 1 0"
                     range="0.0 2.23" damping="0.5" armature="0.01"/>
              <geom type="cylinder" size="0.045 0.035" euler="1.5708 0 0" material="right_mat"/>
              <geom type="capsule"  size="0.022" fromto="0 0 -0.02 0 0 -0.245" material="right_mat"/>

              <body name="right_ankle_cross_link" pos="0.0002 -0.0002 -0.24519">
                <inertial pos="0 0 0" mass="0.039"
                          fullinertia="0.000002 0.000004 0.000004 0 0 0"/>
                <joint name="Right_Ankle_Pitch" type="hinge" axis="0 1 0"
                       range="-0.87 0.345" damping="0.3" armature="0.005"/>

                <body name="right_foot_link" pos="0 0 0">
                  <inertial pos="0.011662 -0.000244 -0.014364" mass="0.494"
                            fullinertia="0.000261 0.001170 0.001213 -0.0000003 -0.000115 0.0000002"/>
                  <joint name="Right_Ankle_Roll" type="hinge" axis="1 0 0"
                         range="-0.345 0.345" damping="0.3" armature="0.005"/>
                  <geom name="right_foot_geom" type="box" size="0.10 0.045 0.020"
                        pos="0.04 0 -0.025" material="foot_mat"
                        condim="3" friction="0.8 0.02 0.01"/>
                  <site name="right_foot_site" pos="0.04 0 -0.025" size="0.005"/>
                </body>
              </body>
            </body>
          </body>
        </body>
      </body>

    </body>
  </worldbody>

  <!-- Actuators — ctrl ∈ [-1, 1]; gear = max torque (Nm) from URDF effort limits -->
  <actuator>
    <motor name="Left_Hip_Pitch"    joint="Left_Hip_Pitch"    gear="30"/>
    <motor name="Left_Hip_Roll"     joint="Left_Hip_Roll"     gear="35"/>
    <motor name="Left_Hip_Yaw"      joint="Left_Hip_Yaw"      gear="20"/>
    <motor name="Left_Knee_Pitch"   joint="Left_Knee_Pitch"   gear="40"/>
    <motor name="Left_Ankle_Pitch"  joint="Left_Ankle_Pitch"  gear="20"/>
    <motor name="Left_Ankle_Roll"   joint="Left_Ankle_Roll"   gear="20"/>
    <motor name="Right_Hip_Pitch"   joint="Right_Hip_Pitch"   gear="30"/>
    <motor name="Right_Hip_Roll"    joint="Right_Hip_Roll"    gear="35"/>
    <motor name="Right_Hip_Yaw"     joint="Right_Hip_Yaw"     gear="20"/>
    <motor name="Right_Knee_Pitch"  joint="Right_Knee_Pitch"  gear="40"/>
    <motor name="Right_Ankle_Pitch" joint="Right_Ankle_Pitch" gear="20"/>
    <motor name="Right_Ankle_Roll"  joint="Right_Ankle_Roll"  gear="20"/>
  </actuator>

  <sensor>
    <accelerometer  name="trunk_accel"     site="imu"/>
    <gyro           name="trunk_gyro"      site="imu"/>
    <touch          name="left_foot_touch" site="left_foot_site"/>
    <touch          name="right_foot_touch" site="right_foot_site"/>
  </sensor>

  <keyframe>
    <key name="stand"
         qpos="0 0 0.55  1 0 0 0
               0 0 0  0.5 -0.25 0
               0 0 0  0.5 -0.25 0"
         ctrl="0 0 0 0 0 0  0 0 0 0 0 0"/>
  </keyframe>
</mujoco>
"""

# Compile-time check — will throw if XML is malformed
import mujoco as _mj
_test_model = _mj.MjModel.from_xml_string(K1_MJCF)
print(f"✓ MJCF parsed OK  |  nq={_test_model.nq}  nv={_test_model.nv}  nu={_test_model.nu}")
del _test_model

In [ ]:
import numpy as np
import mujoco
import gymnasium as gym
from gymnasium import spaces


class K1WalkEnv(gym.Env):
    """
    MuJoCo locomotion environment for the Booster K1 humanoid.

    Observation (48 dims):
        base_lin_vel(3) | base_ang_vel(3) | proj_gravity(3) |
        cmd_vel(3) | joint_pos_offset(12) | joint_vel(12) | prev_action(12)

    Action (12 dims): normalised torques ∈ [-1, 1] for the 12 leg joints.
        Actual torque = action × gear_ratio  (gear defined in MJCF)

    Termination:
        base_height < 0.30 m  OR  tilted more than ~60° from vertical
    """

    metadata = {"render_modes": ["rgb_array"]}

    # Joint order must match MJCF <actuator> order
    LEG_JOINT_NAMES = [
        "Left_Hip_Pitch",  "Left_Hip_Roll",  "Left_Hip_Yaw",
        "Left_Knee_Pitch", "Left_Ankle_Pitch","Left_Ankle_Roll",
        "Right_Hip_Pitch", "Right_Hip_Roll",  "Right_Hip_Yaw",
        "Right_Knee_Pitch","Right_Ankle_Pitch","Right_Ankle_Roll",
    ]

    # Comfortable standing pose — both knees gently bent
    DEFAULT_QPOS = np.array([
         0.0, 0.0, 0.0,   0.5, -0.25, 0.0,   # left  hip-p/r/y, knee, ank-p/r
         0.0, 0.0, 0.0,   0.5, -0.25, 0.0,   # right hip-p/r/y, knee, ank-p/r
    ])

    def __init__(self, render_mode=None):
        super().__init__()
        self.model = mujoco.MjModel.from_xml_string(K1_MJCF)
        self.data  = mujoco.MjData(self.model)

        self.render_mode = render_mode
        self._renderer   = None

        n = len(self.LEG_JOINT_NAMES)
        self.n_joints    = n
        self.joint_ids   = [
            mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_JOINT, nm)
            for nm in self.LEG_JOINT_NAMES
        ]

        obs_dim = 3 + 3 + 3 + 3 + n + n + n   # 48
        self.observation_space = spaces.Box(-np.inf, np.inf, (obs_dim,), np.float32)
        self.action_space      = spaces.Box(-1.0, 1.0, (n,), np.float32)

        # Simulation timing
        self.sim_dt     = self.model.opt.timestep   # 0.005 s
        self.control_dt = 0.02                       # 50 Hz
        self.n_substeps = int(round(self.control_dt / self.sim_dt))

        # Sensor IDs
        def _sid(name):
            return mujoco.mj_name2id(self.model, mujoco.mjtObj.mjOBJ_SENSOR, name)
        self._lfoot_sid = _sid("left_foot_touch")
        self._rfoot_sid = _sid("right_foot_touch")

        # Episode state (reset in reset())
        self._prev_action   = np.zeros(n)
        self._cmd_vel       = np.zeros(3)
        self._feet_air_time = np.zeros(2)
        self._last_contacts = np.zeros(2, dtype=bool)
        self._step_count    = 0
        self.max_episode_steps = 1000

    # ── Gym API ────────────────────────────────────────────────────────────────

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        mujoco.mj_resetData(self.model, self.data)

        # Set floating-base position (freejoint: qpos[0:7])
        self.data.qpos[2]   = 0.55          # height
        self.data.qpos[3]   = 1.0           # qw  (upright)
        self.data.qpos[4:7] = 0.0           # qx qy qz

        # Set leg joint angles to default standing pose
        for i, jid in enumerate(self.joint_ids):
            self.data.qpos[self.model.jnt_qposadr[jid]] = self.DEFAULT_QPOS[i]

        mujoco.mj_forward(self.model, self.data)

        # Command velocity
        eval_mode = options.get("eval_mode", False) if options else False
        if eval_mode:
            self._cmd_vel = np.array([0.5, 0.0, 0.0])
        else:
            rng = self.np_random
            self._cmd_vel = np.array([
                rng.uniform(-0.3, 1.0),
                rng.uniform(-0.3,  0.3),
                rng.uniform(-0.5,  0.5),
            ])

        self._prev_action   = np.zeros(self.n_joints)
        self._feet_air_time = np.zeros(2)
        self._last_contacts = np.zeros(2, dtype=bool)
        self._step_count    = 0

        return self._obs(), {}

    def step(self, action: np.ndarray):
        # Write controls (normalised [-1,1]; MJCF gear scales to Nm)
        self.data.ctrl[:self.n_joints] = np.clip(action, -1.0, 1.0)

        # Step physics at 200 Hz, control at 50 Hz
        for _ in range(self.n_substeps):
            mujoco.mj_step(self.model, self.data)

        obs    = self._obs()
        reward, info = self._reward(action)
        terminated   = self._is_terminated()
        truncated    = self._step_count >= self.max_episode_steps

        self._prev_action = action.copy()
        self._step_count += 1

        return obs, reward, terminated, truncated, info

    def render(self):
        if self.render_mode != "rgb_array":
            return None
        if self._renderer is None:
            self._renderer = mujoco.Renderer(self.model, height=480, width=640)
        self._renderer.update_scene(self.data, camera="track")
        return self._renderer.render()

    def close(self):
        if self._renderer is not None:
            self._renderer.close()
            self._renderer = None

    # ── Internal helpers ───────────────────────────────────────────────────────

    def _obs(self) -> np.ndarray:
        quat = self.data.qpos[3:7]   # [w, x, y, z]

        lin_vel   = self._w2b(self.data.qvel[:3], quat)
        ang_vel   = self._w2b(self.data.qvel[3:6], quat)
        grav_body = self._w2b(np.array([0.0, 0.0, -1.0]), quat)

        qpos = np.array([
            self.data.qpos[self.model.jnt_qposadr[jid]]
            for jid in self.joint_ids
        ]) - self.DEFAULT_QPOS

        qvel = np.array([
            self.data.qvel[self.model.jnt_dofadr[jid]]
            for jid in self.joint_ids
        ])

        return np.concatenate([
            lin_vel, ang_vel, grav_body, self._cmd_vel,
            qpos, qvel, self._prev_action,
        ]).astype(np.float32)

    def _reward(self, action):
        quat    = self.data.qpos[3:7]
        lin_vel = self._w2b(self.data.qvel[:3], quat)
        ang_vel = self._w2b(self.data.qvel[3:6], quat)
        grav_b  = self._w2b(np.array([0.0, 0.0, -1.0]), quat)

        # 1. Linear velocity tracking  (main task)
        r_lin = float(np.exp(-np.sum((self._cmd_vel[:2] - lin_vel[:2])**2) / 0.25)) * 2.0

        # 2. Yaw-rate tracking
        r_ang = float(np.exp(-(self._cmd_vel[2] - ang_vel[2])**2 / 0.25)) * 0.5

        # 3. Stability penalties
        r_lin_z  = -0.20 * float(lin_vel[2]**2)
        r_ang_xy = -0.05 * float(np.sum(ang_vel[:2]**2))
        r_orient = -0.20 * float(np.sum(grav_b[:2]**2))
        r_height = -0.10 * float((self.data.qpos[2] - 0.50)**2)

        # 4. Control effort
        r_energy = -1e-4 * float(np.sum(self.data.ctrl[:self.n_joints]**2))

        # 5. Action smoothness
        r_smooth = -0.02 * float(np.sum((action - self._prev_action)**2))

        # 6. Feet air-time (encourages periodic gait)
        lc = self._foot_contact(0)
        rc = self._foot_contact(1)
        contacts = np.array([lc, rc], dtype=float)
        self._feet_air_time += self.control_dt
        self._feet_air_time *= (1.0 - contacts)           # reset counter on landing
        first_contact = contacts.astype(bool) & ~self._last_contacts
        cmd_speed = float(np.linalg.norm(self._cmd_vel[:2]))
        r_airtime = (0.25 * float(np.sum((self._feet_air_time - 0.5) * first_contact))
                     * (cmd_speed > 0.1))
        self._last_contacts = contacts.astype(bool)

        reward = r_lin + r_ang + r_lin_z + r_ang_xy + r_orient + r_height + r_energy + r_smooth + r_airtime

        info = {
            "r_lin_vel": r_lin, "r_ang_vel": r_ang,
            "r_orient":  r_orient, "r_height": r_height,
            "r_energy":  r_energy, "r_airtime": r_airtime,
            "base_height": float(self.data.qpos[2]),
            "base_vx":   float(lin_vel[0]),
        }
        return reward, info

    def _is_terminated(self) -> bool:
        if self.data.qpos[2] < 0.30:
            return True
        grav_b = self._w2b(np.array([0.0, 0.0, -1.0]), self.data.qpos[3:7])
        return bool(grav_b[2] > -0.5)     # tilted > ~60°

    def _foot_contact(self, idx: int) -> bool:
        sid  = self._lfoot_sid if idx == 0 else self._rfoot_sid
        adr  = self.model.sensor_adr[sid]
        return float(self.data.sensordata[adr]) > 1.0

    @staticmethod
    def _w2b(v: np.ndarray, q: np.ndarray) -> np.ndarray:
        """Rotate vector v from world frame to body frame given quaternion q=[w,x,y,z]."""
        w, x, y, z = q
        R = np.array([
            [1 - 2*(y*y + z*z),  2*(x*y - w*z),  2*(x*z + w*y)],
            [    2*(x*y + w*z),  1 - 2*(x*x + z*z), 2*(y*z - w*x)],
            [    2*(x*z - w*y),  2*(y*z + w*x),  1 - 2*(x*x + y*y)],
        ])
        return R.T @ v

In [ ]:
import numpy as np
from stable_baselines3.common.callbacks import BaseCallback
import wandb, imageio, os

class WandbVideoCallback(BaseCallback):
    """
    Renders K1WalkEnv at fixed intervals and uploads an MP4 to W&B.
    Add to your callbacks list alongside WandbCallback.
    """

    def __init__(self, eval_env_fn, log_freq: int = 500_000,
                 n_steps: int = 500, video_fps: int = 50, verbose: int = 0):
        super().__init__(verbose)
        self._eval_env_fn = eval_env_fn   # no-arg callable → K1WalkEnv(render_mode="rgb_array")
        self._log_freq    = log_freq
        self._n_steps     = n_steps
        self._video_fps   = video_fps
        self._last_log    = 0

    def _on_step(self) -> bool:
        if self.num_timesteps - self._last_log < self._log_freq:
            return True
        self._last_log = self.num_timesteps

        env = self._eval_env_fn()
        obs, _ = env.reset(options={"eval_mode": True})
        frames, total_r = [], 0.0

        vec_norm = None
        if hasattr(self.training_env, "normalize_obs"):
            vec_norm = self.training_env

        for _ in range(self._n_steps):
            if vec_norm is not None:
                obs_in = vec_norm.normalize_obs(obs[np.newaxis])[0]
            else:
                obs_in = obs
            action, _ = self.model.predict(obs_in, deterministic=True)
            obs, r, terminated, truncated, _ = env.step(action)
            total_r += r
            frame = env.render()
            if frame is not None:
                frames.append(frame)
            if terminated or truncated:
                break

        env.close()

        if frames and wandb.run is not None:
            path = f"/tmp/k1_walk_{self.num_timesteps}.mp4"
            imageio.mimwrite(path, frames, fps=self._video_fps)
            wandb.log({
                "eval/video":   wandb.Video(path, fps=self._video_fps, format="mp4"),
                "eval/ep_reward_video": total_r,
            }, step=self.num_timesteps)
            if self.verbose:
                print(f"[WandbVideoCallback] step={self.num_timesteps}  reward={total_r:.2f}  frames={len(frames)}")

        return True


# ── Quick environment sanity-check ────────────────────────────────────────────
_env = K1WalkEnv()
_obs, _ = _env.reset()
print(f"obs shape  : {_obs.shape}")
print(f"action shape: {_env.action_space.shape}")
print(f"obs sample : {_obs[:6].round(3)}")

_total = 0.0
for _step in range(300):
    _act = _env.action_space.sample() * 0.1
    _obs, _r, _done, _trunc, _ = _env.step(_act)
    _total += _r
    if _done or _trunc:
        break
print(f"Random-policy steps={_step+1}  total_reward={_total:.2f}")
_env.close()
print("✓ Environment OK")

In [ ]:
# ── PPO hyperparameters ───────────────────────────────────────────────────────
TRAIN_CFG = dict(
    # PPO core
    learning_rate   = 3e-4,
    n_steps         = 2048,   # steps per env per update
    batch_size      = 256,
    n_epochs        = 10,
    gamma           = 0.99,
    gae_lambda      = 0.95,
    clip_range      = 0.2,
    ent_coef        = 0.005,  # mild entropy bonus
    vf_coef         = 0.5,
    max_grad_norm   = 1.0,
    # Training length — increase for better performance
    total_timesteps = 30_000_000,
)

print("Training config:")
for k, v in TRAIN_CFG.items():
    print(f"  {k:20s} = {v}")

In [ ]:
import os
import numpy as np
import torch.nn as nn
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import SubprocVecEnv, DummyVecEnv, VecNormalize
from stable_baselines3.common.callbacks import EvalCallback, CheckpointCallback
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.utils import set_random_seed
from wandb.integration.sb3 import WandbCallback
import wandb

# ── Env factory ──────────────────────────────────────────────────────────────
def make_env(rank: int, seed: int = 0):
    def _init():
        env = Monitor(K1WalkEnv())
        env.reset(seed=seed + rank)
        return env
    set_random_seed(seed + rank)
    return _init

# ── Training ─────────────────────────────────────────────────────────────────
N_ENVS = min(int(os.cpu_count() or 4), 8)
print(f"Spawning {N_ENVS} parallel environments …")

run = wandb.init(
    project=WANDB_PROJECT,
    entity=WANDB_ENTITY,
    name=EXPERIMENT_NAME,
    config=TRAIN_CFG,
    sync_tensorboard=True,
    monitor_gym=True,
    save_code=True,
)

# Vectorised training env
try:
    vec_env = SubprocVecEnv([make_env(i) for i in range(N_ENVS)], start_method="fork")
except Exception:
    print("SubprocVecEnv failed → using DummyVecEnv")
    vec_env = DummyVecEnv([make_env(i) for i in range(N_ENVS)])
vec_env = VecNormalize(vec_env, norm_obs=True, norm_reward=True, clip_obs=10.0)

# Separate eval env (frozen normalisation)
eval_env = DummyVecEnv([make_env(0, seed=999)])
eval_env = VecNormalize(eval_env, norm_obs=True, norm_reward=False, clip_obs=10.0, training=False)

# Policy network: 3-layer MLP for actor and critic
policy_kwargs = dict(
    net_arch=[dict(pi=[512, 256, 128], vf=[512, 256, 128])],
    activation_fn=nn.ELU,
)

model = PPO(
    "MlpPolicy", vec_env,
    learning_rate   = TRAIN_CFG["learning_rate"],
    n_steps         = TRAIN_CFG["n_steps"],
    batch_size      = TRAIN_CFG["batch_size"],
    n_epochs        = TRAIN_CFG["n_epochs"],
    gamma           = TRAIN_CFG["gamma"],
    gae_lambda      = TRAIN_CFG["gae_lambda"],
    clip_range      = TRAIN_CFG["clip_range"],
    ent_coef        = TRAIN_CFG["ent_coef"],
    vf_coef         = TRAIN_CFG["vf_coef"],
    max_grad_norm   = TRAIN_CFG["max_grad_norm"],
    policy_kwargs   = policy_kwargs,
    verbose         = 1,
    tensorboard_log = f"runs/{run.id}",
)

os.makedirs(f"models/{run.id}/best", exist_ok=True)
os.makedirs(f"checkpoints/{run.id}", exist_ok=True)

callbacks = [
    WandbCallback(
        gradient_save_freq = 20_000,
        model_save_path    = f"models/{run.id}",
        verbose            = 2,
    ),
    # Uploads a rendered video to W&B every 500 k steps
    WandbVideoCallback(
        eval_env_fn = lambda: K1WalkEnv(render_mode="rgb_array"),
        log_freq    = 500_000,
        n_steps     = 400,
        video_fps   = 50,
        verbose     = 1,
    ),
    EvalCallback(
        eval_env,
        best_model_save_path = f"models/{run.id}/best",
        log_path             = f"logs/{run.id}",
        eval_freq            = max(100_000 // N_ENVS, 1),
        n_eval_episodes      = 5,
        deterministic        = True,
        verbose              = 1,
    ),
    CheckpointCallback(
        save_freq   = max(500_000 // N_ENVS, 1),
        save_path   = f"checkpoints/{run.id}",
        name_prefix = "k1_walk",
    ),
]

model.learn(
    total_timesteps = TRAIN_CFG["total_timesteps"],
    callback        = callbacks,
    progress_bar    = True,
)

# Save final artefacts
model.save(f"models/{run.id}/final_model")
vec_env.save(f"models/{run.id}/vec_normalize.pkl")
print(f"✓ Training complete — artefacts saved under models/{run.id}/")
run.finish()

In [ ]:
import imageio, os
import numpy as np
from IPython.display import Video, display
from stable_baselines3 import PPO
from stable_baselines3.common.vec_env import DummyVecEnv, VecNormalize
from stable_baselines3.common.monitor import Monitor
import wandb


def evaluate_policy(run_id: str, n_steps: int = 600, n_episodes: int = 3,
                    log_to_wandb: bool = True):
    """
    Load the best checkpoint, run n_episodes, render to MP4, and
    (optionally) upload the video + per-episode stats to W&B.
    """
    model_path   = f"models/{run_id}/best/best_model"
    vecnorm_path = f"models/{run_id}/vec_normalize.pkl"

    # Rebuild normalised eval env
    eval_env = DummyVecEnv([lambda: Monitor(K1WalkEnv())])
    eval_env = VecNormalize.load(vecnorm_path, eval_env)
    eval_env.training    = False
    eval_env.norm_reward = False

    model = PPO.load(model_path, env=eval_env)

    all_rewards, all_frames = [], []

    for ep in range(n_episodes):
        render_env = K1WalkEnv(render_mode="rgb_array")
        obs_raw, _ = render_env.reset(options={"eval_mode": True})
        ep_r, frames = 0.0, []

        for step in range(n_steps):
            obs_n         = eval_env.normalize_obs(obs_raw[np.newaxis])[0]
            action, _     = model.predict(obs_n, deterministic=True)
            obs_raw, r, terminated, truncated, _ = render_env.step(action)
            ep_r += r
            frame = render_env.render()
            if frame is not None:
                frames.append(frame)
            if terminated or truncated:
                break

        render_env.close()
        all_rewards.append(ep_r)
        all_frames.extend(frames)
        print(f"  Episode {ep+1}: reward={ep_r:.2f}  steps={step+1}")

    mean_r = float(np.mean(all_rewards))
    std_r  = float(np.std(all_rewards))
    print(f"\nMean reward: {mean_r:.2f} ± {std_r:.2f}")

    # Save video locally
    video_path = "k1_walk_eval.mp4"
    if all_frames:
        imageio.mimwrite(video_path, all_frames[:900], fps=50)
        print(f"Video saved → {video_path}")

    # Upload to W&B
    if log_to_wandb and all_frames and wandb.run is not None:
        wandb.log({
            "eval/mean_reward":  mean_r,
            "eval/std_reward":   std_r,
            "eval/final_video":  wandb.Video(video_path, fps=50, format="mp4"),
        })
        print("✓ Video + stats logged to W&B")

    # Display in notebook
    if all_frames:
        try:
            import mediapy
            mediapy.show_video(all_frames[:300], fps=50, width=480)
        except Exception:
            display(Video(video_path, embed=True, width=480))

    eval_env.close()


# ── Run after training ────────────────────────────────────────────────────────
# evaluate_policy(run.id, log_to_wandb=True)